# META-CXR — Stage-2 Evaluation Notebook (Kaggle, 2×T4 GPU)

Đánh giá pipeline sinh báo cáo X-quang ngực **end-to-end** trên MIMIC-CXR **p10 preprocessed split**:

```text
Image -> BioViL-T / PubMedCLIP -> Q-Former + MHCAC -> 14-abnormality findings -> LoRA-Vicuna-7B -> Generated Report
```

**Metrics output**: BLEU-1/2/3/4, METEOR, ROUGE-L (qua `MIMICEvalCap` có sẵn trong repo) + CheXpert-style classification F1 cho 14 abnormalities.

---

## Resources & Kaggle datasets

Attach thủ công qua **Notebook Settings -> Add Data**:

1. **mimic-cxr-jpg-lite** — chest X-ray JPG files + raw metadata CSVs:
   `mimic-cxr-2.0.0-chexpert.csv`, `mimic-cxr-2.0.0-metadata.csv`
2. **mimic-cxr-p10-processed** — split sau preprocessing:
   `p10_train.csv`, `p10_val.csv`, `p10_test.csv`, `all_data.csv`
   (notebook cũng chấp nhận alias `train.csv`, `val.csv`, `test.csv`)
3. **meta-cxr-checkpoints** — Q-Former + MHCAC stage-1 weights, file `.pth`
4. *(Optional)* **lmsys/vicuna-7b-v1.3** — nếu có dataset bundled, notebook ưu tiên dùng thay vì HF download

Không cần attach dataset `mimic-cxr-reported` cho notebook eval này vì ground truth findings được đọc trực tiếp từ CSV đã preprocessing.

### Kaggle environment

- Accelerator: GPU **T4 × 2**
- Internet: **ON** nếu cần tải dependency / Vicuna / pretrained encoder

---

## Tunable parameters (Cell 5)

`EVAL_LIMIT`, `EVAL_SPLIT`, `BATCH_SIZE_IMG`, `NUM_BEAMS`, `MAX_NEW_TOKENS`, `RUN_CHEXPERT_F1`, `TRY_TEXT_LABELER`.

## Expected runtime (2×T4)

| Mode | Samples | Time |
|------|---------|------|
| Smoke test | 4 | ~1 phút |
| p10 test split | ~2,000 | ~1-2 giờ |
| p10 train split | ~18,000 | nhiều giờ, chỉ nên chạy khi thật cần |

> Bắt đầu với `EVAL_LIMIT = 4` để smoke test đường dẫn dataset/checkpoint trước khi chạy full split.

---


In [ ]:
# Cell 1 — Install dependencies pinned to versions verified for this repo
import subprocess, sys, os

PIP_INSTALL = [
    "transformers==4.44.2",
    "timm>=0.9.0",
    "loralib==0.1.1",
    "git+https://github.com/huggingface/peft.git@e536616",
    "omegaconf==2.3.0",
    "nltk>=3.9",
    "pycocoevalcap",
    "hi-ml-multimodal",
    "huggingface_hub",
    "scikit-image",
    "scikit-learn",
    "wandb",
]
for pkg in PIP_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# --- Detect Java for METEOR (and optional CheXpert labeler) ---
def _detect_java_home():
    try:
        which = subprocess.check_output(["which", "java"]).decode().strip()
        real = subprocess.check_output(["readlink", "-f", which]).decode().strip()
        return real.replace("/bin/java", "")
    except subprocess.CalledProcessError:
        return "/usr/lib/jvm/java-8-openjdk-amd64/jre"

JAVA_HOME = _detect_java_home()
JAVA_PATH = f"{JAVA_HOME}/bin:"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_PATH + os.environ.get("PATH", "")
print(f"JAVA_HOME = {JAVA_HOME}")

# --- NLTK data for METEOR ---
import nltk
for resource in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"NLTK download {resource!r} failed: {e}")
print("Dependencies installed.")


In [ ]:
# Cell 2 — Kaggle credentials (only needed if Cell 11 should push results back)
import os, json

HAVE_KAGGLE_CREDS = False
KAGGLE_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    KAGGLE_USER = secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY")
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
        json.dump({"username": KAGGLE_USER, "key": KAGGLE_KEY}, f)
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
    HAVE_KAGGLE_CREDS = True
    print(f"Kaggle credentials configured for user {KAGGLE_USER!r}")
except Exception as e:
    print(f"Kaggle credentials not configured ({e!r}) — Cell 11 push-back will be skipped.")


In [ ]:
# Cell 3 — Clone META-CXR repo
import os, subprocess, sys

REPO_URL = os.environ.get("REPO_URL", "https://github.com/minhphuong150505/Meta-CXR-Kaggle.git")
REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"Repo at {REPO_DIR} @ {head}")


In [ ]:
# Cell 4 — Detect & validate mounted Kaggle datasets for p10 preprocessed eval
import os
from pathlib import Path

MOUNT_ROOTS = ["/kaggle/input/datasets", "/kaggle/input"]


def iter_dataset_candidates(max_depth=2):
    """Yield dataset roots mounted either as /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>."""
    for root in MOUNT_ROOTS:
        root_path = Path(root)
        if not root_path.is_dir():
            continue
        stack = [(root_path, 0)]
        while stack:
            path, depth = stack.pop(0)
            yield path
            if depth >= max_depth:
                continue
            try:
                children = sorted([p for p in path.iterdir() if p.is_dir()])
            except OSError as exc:
                print(f"Skip {path}: {exc}")
                continue
            stack.extend((child, depth + 1) for child in children)


def _has_markers(candidate, marker_files):
    for marker in marker_files:
        if (candidate / marker).exists():
            continue
        if not list(candidate.rglob(marker)):
            return False
    return True


def find_dataset(slug_keywords, marker_files):
    """Find first mounted dataset whose path contains a slug keyword and marker files."""
    for candidate in iter_dataset_candidates():
        haystack = str(candidate).lower()
        if not any(kw in haystack for kw in slug_keywords):
            continue
        if _has_markers(candidate, marker_files):
            return str(candidate)
    return None


def pick_file(root, names, required=True):
    root_path = Path(root)
    for name in names:
        direct = root_path / name
        if direct.exists():
            return str(direct)
        matches = sorted(root_path.rglob(name))
        if matches:
            return str(matches[0])
    if required:
        raise FileNotFoundError(f"None of {names} found under {root}")
    return ""


MIMIC_IMG_ROOT = find_dataset(
    ["mimic-cxr-jpg-lite", "mimic-cxr-jpg", "mimic_cxr_jpg"],
    ["mimic-cxr-2.0.0-chexpert.csv"],
)
assert MIMIC_IMG_ROOT, "Mount mimic-cxr-jpg-lite via Notebook Settings -> Add Data"

PROCESSED_ROOT = (
    find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["p10_test.csv"])
    or find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["test.csv"])
    or find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["all_data.csv"])
)
assert PROCESSED_ROOT, "Mount mimic-cxr-p10-processed via Notebook Settings -> Add Data"

PROCESSED_TRAIN_CSV = pick_file(PROCESSED_ROOT, ["p10_train.csv", "train.csv"])
PROCESSED_VAL_CSV = pick_file(PROCESSED_ROOT, ["p10_val.csv", "val.csv"])
PROCESSED_TEST_CSV = pick_file(PROCESSED_ROOT, ["p10_test.csv", "test.csv"])
PROCESSED_ALL_CSV = pick_file(PROCESSED_ROOT, ["p10_all_data.csv", "all_data.csv"], required=False)
PROCESSED_SPLIT_CSV = pick_file(PROCESSED_ROOT, ["mimic-cxr-2.0.0-split-p10.csv"], required=False)

CKPT_ROOT = find_dataset(
    ["meta-cxr-checkpoints", "meta_cxr_checkpoints", "meta-cxr-checkpoint", "mimic_cxr_checkpoint"],
    [],
)
assert CKPT_ROOT, "Mount meta-cxr-checkpoints or the project checkpoint dataset"

# Find the best Q-Former checkpoint: prefer best > last > highest epoch
pth_files = list(Path(CKPT_ROOT).rglob("*.pth"))
assert pth_files, f"No .pth files in {CKPT_ROOT}"


def ckpt_priority(p: Path):
    name = p.name
    if name == "checkpoint_best.pth":
        return (0, 0)
    if name == "checkpoint_last.pth":
        return (1, 0)
    if name.startswith("checkpoint_") and name.endswith(".pth"):
        try:
            return (2, -int(name.replace("checkpoint_", "").replace(".pth", "")))
        except ValueError:
            return (3, 0)
    return (4, 0)


QFORMER_CKPT = str(sorted(pth_files, key=ckpt_priority)[0])

# Vicuna dataset (optional)
VICUNA_ROOT = None
for candidate in iter_dataset_candidates():
    e_lower = str(candidate).lower()
    if "vicuna" in e_lower and "7b" in e_lower:
        cfg_candidates = list(Path(candidate).rglob("config.json"))
        if cfg_candidates:
            VICUNA_ROOT = str(cfg_candidates[0].parent)
            break

NEED_HF_DOWNLOAD = VICUNA_ROOT is None
print("--- Dataset summary ---")
print(f"  MIMIC images      : {MIMIC_IMG_ROOT}")
print(f"  Processed p10 root: {PROCESSED_ROOT}")
print(f"  train split CSV   : {PROCESSED_TRAIN_CSV}")
print(f"  val split CSV     : {PROCESSED_VAL_CSV}")
print(f"  test split CSV    : {PROCESSED_TEST_CSV}")
print(f"  Ckpt picked       : {QFORMER_CKPT}")
print(f"  Vicuna local      : {VICUNA_ROOT or '(none -> will HF-download in Cell 6)'}")


In [ ]:
# Cell 5 — Tunable evaluation parameters
EVAL_LIMIT       = None       # None = full split; e.g. 4 (smoke), 500 (subset)
EVAL_SPLIT       = "test"     # "train", "val", or "test"
NUM_BEAMS        = 1
MAX_NEW_TOKENS   = 300
BATCH_SIZE_IMG   = 8          # batch size for BLIP image forward (lower to 4 if OOM)
RUN_CHEXPERT_F1  = True       # classification F1 over 14 abnormalities from Q-Former logits
TRY_TEXT_LABELER = False      # Java CheXpert labeler on generated text (fragile, off by default)

# Keep these aligned with the encoder topology used during training.
ENCODER_BIOVIL     = True
ENCODER_PUBMEDCLIP = True
ENCODER_SWIN       = False

print(f"EVAL_LIMIT={EVAL_LIMIT}  EVAL_SPLIT={EVAL_SPLIT}  BATCH_SIZE_IMG={BATCH_SIZE_IMG}  num_beams={NUM_BEAMS}")
print(f"encoders: biovil={ENCODER_BIOVIL}, pubmedclip={ENCODER_PUBMEDCLIP}, swin={ENCODER_SWIN}")


In [ ]:
# Cell 6 — Write configs/env_config.yaml + eval_config.yaml; optionally download Vicuna
import os
from pathlib import Path
from omegaconf import OmegaConf


def _resolve_or_empty(root, name):
    matches = list(Path(root).rglob(name))
    return str(matches[0]) if matches else ""


split_csv = PROCESSED_SPLIT_CSV or _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-split.csv")
chexpert_csv = _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-chexpert.csv")
metadata_csv = _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-metadata.csv")
reports_csv = PROCESSED_ALL_CSV or PROCESSED_TEST_CSV

assert chexpert_csv, f"Missing mimic-cxr-2.0.0-chexpert.csv under {MIMIC_IMG_ROOT}"
assert metadata_csv, f"Missing mimic-cxr-2.0.0-metadata.csv under {MIMIC_IMG_ROOT}"

OUT_DIR = "/kaggle/working/eval_output"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("configs", exist_ok=True)

env_cfg = {
    "paths": {
        "data_root": "/kaggle/input",
        "mimic_cxr_jpg_root": MIMIC_IMG_ROOT,
        "split_csv": split_csv,
        "reports_csv": reports_csv,
        "chexpert_csv": chexpert_csv,
        "metadata_csv": metadata_csv,
        "processed_dir": PROCESSED_ROOT,
        "processed_train_csv": PROCESSED_TRAIN_CSV,
        "processed_val_csv": PROCESSED_VAL_CSV,
        "processed_test_csv": PROCESSED_TEST_CSV,
        "output_dir": OUT_DIR,
        "checkpoint_dir": OUT_DIR,
    },
    "wandb": {"entity": "", "project": "meta-cxr-eval"},
    "java": {"home": JAVA_HOME, "path": JAVA_PATH},
}
with open("configs/env_config.yaml", "w") as f:
    OmegaConf.save(OmegaConf.create(env_cfg), f)
print("Wrote configs/env_config.yaml")
print(f"Evaluation split source: {EVAL_SPLIT} -> {env_cfg['paths'][f'processed_{EVAL_SPLIT}_csv']}")

# Load base inference config and override for eval
base_cfg = OmegaConf.load("pretraining/configs/blip2_pretrain_stage1_emb.yaml")
base_cfg.model.finetuned = QFORMER_CKPT
base_cfg.model.load_finetuned = True
base_cfg.model.encoders = OmegaConf.create({
    "biovil": bool(ENCODER_BIOVIL),
    "pubmedclip": bool(ENCODER_PUBMEDCLIP),
    "swin": bool(ENCODER_SWIN),
})
base_cfg.model.llm = OmegaConf.create({"lora_path": "checkpoints/lora-vicuna-7b-report-20250621"})
base_cfg.run.evaluate = True
base_cfg.run.task = "image_text_pretrain_eval"
base_cfg.run.batch_size_eval = BATCH_SIZE_IMG
base_cfg.run.num_beams = NUM_BEAMS
base_cfg.run.max_len = MAX_NEW_TOKENS
base_cfg.run.output_dir = OUT_DIR
base_cfg.run.distributed = False
base_cfg.run.world_size = 1
# Remove resume_ckpt_path so runner doesn't try to load training state
if "resume_ckpt_path" in base_cfg.run:
    del base_cfg.run.resume_ckpt_path

EVAL_CFG_PATH = "configs/eval_config.yaml"
with open(EVAL_CFG_PATH, "w") as f:
    OmegaConf.save(base_cfg, f)
print(f"Wrote {EVAL_CFG_PATH}")

# HF download Vicuna if no Kaggle dataset attached
if NEED_HF_DOWNLOAD:
    from huggingface_hub import snapshot_download
    VICUNA_ROOT = "/kaggle/working/vicuna-7b-v1.3"
    if not os.path.exists(os.path.join(VICUNA_ROOT, "config.json")):
        print("Downloading Vicuna-7B-v1.3 from HuggingFace (~14 GB, 10-15 min)...")
        snapshot_download(
            repo_id="lmsys/vicuna-7b-v1.3",
            local_dir=VICUNA_ROOT,
            resume_download=True,
        )
    print(f"Vicuna ready at: {VICUNA_ROOT}")
else:
    print(f"Vicuna already mounted at: {VICUNA_ROOT}")


In [ ]:
# Cell 7 — Build Q-Former (BLIP) + LoRA-Vicuna
# (init helpers copied from inference.py to avoid Gradio import side effects at module load.)
import sys, argparse
import torch
from torch import nn
from transformers import LlamaTokenizer
from peft import PeftModelForCausalLM

# Patch sys.argv so the in-repo Config(parse_args()) sees our eval config path
sys.argv = ["eval_kaggle", "--cfg-path", EVAL_CFG_PATH]

from model.lavis.common.config import Config


def _parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--cfg-path", required=True)
    p.add_argument("--local_rank", type=int, default=0)
    p.add_argument("--options", nargs="+")
    return p.parse_args()


cfg = Config(_parse_args())

# Register all LAVIS components (must precede tasks.setup_task)
import model.lavis.tasks as tasks                  # noqa: E402
from model.lavis.datasets.builders import *        # noqa: E402, F401, F403
from model.lavis.models import *                   # noqa: E402, F401, F403
from model.lavis.processors import *               # noqa: E402, F401, F403
from model.lavis.runners import *                  # noqa: E402, F401, F403
from model.lavis.tasks import *                    # noqa: E402, F401, F403
from model.lavis.models.blip2_models.modeling_llama_imgemb import LlamaForCausalLM  # noqa: E402


def init_blip(cfg):
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    return model.to(torch.device("cpu"))


def init_vicuna(vicuna_path):
    tok = LlamaTokenizer.from_pretrained(
        vicuna_path, use_fast=False, truncation_side="left", padding_side="left"
    )
    lang = LlamaForCausalLM.from_pretrained(
        vicuna_path, torch_dtype=torch.float16, device_map="auto"
    )
    tok.pad_token = tok.unk_token
    lang.base_model.img_proj_layer = nn.Linear(
        768, lang.base_model.config.hidden_size
    ).to(lang.base_model.device)
    tok.add_special_tokens({"additional_special_tokens": ["<IMG>"]})
    lang = PeftModelForCausalLM.from_pretrained(
        lang,
        cfg.config.model.llm.lora_path,
        torch_dtype=torch.float16,
        use_ram_optimized_load=False,
    ).half()
    return lang, tok


print("Building BLIP (Q-Former + MHCAC)...")
blip_model = init_blip(cfg).cuda().eval()

print("Building Vicuna + LoRA...")
lang_model, vicuna_tokenizer = init_vicuna(VICUNA_ROOT)
lang_model.eval()

# Sanity check: forward a dummy image
with torch.no_grad():
    dummy = torch.randn(1, 3, 448, 448).cuda()
    out = blip_model.forward_image(dummy)
    logits = out[0]
    qf = out[1]
    print(f"BLIP forward_image OK -> logits {tuple(logits.shape)}  qformer_embs {tuple(qf.shape)}")


In [ ]:
# Cell 8 — Run inference on the eval split
import os, json, time
import numpy as np
import torch
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset

ABN_NAMES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
    "Fracture", "Support Devices",
]


def classify_abnormalities(logits, thresholds_path=None, class_map=None):
    """Mirror of inference.py:classify_abnormalities (re-defined to avoid Gradio import)."""
    if class_map is None:
        class_map = {"uncertain": 2, "positive": 1, "negative": 0}
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)
    probs = torch.softmax(logits, dim=1).tolist()
    categorized = {cls: [] for cls in class_map}

    thresholds_data, use_th = {}, False
    if thresholds_path and os.path.isfile(thresholds_path):
        with open(thresholds_path) as f:
            thresholds_data = json.load(f)
        use_th = True

    for abn, p in zip(ABN_NAMES, probs):
        if abn == "No Finding":
            continue
        if use_th:
            best_cls, best_score = None, 0.0
            for cls, idx in class_map.items():
                t = thresholds_data.get(abn, {}).get(cls, 0.5)
                if p[idx] >= t and p[idx] > best_score:
                    best_cls, best_score = cls, p[idx]
            if best_cls is not None:
                categorized[best_cls].append(abn)
        else:
            idx = int(torch.tensor(p).argmax().item())
            cls = [k for k, v in class_map.items() if v == idx][0]
            categorized[cls].append(abn)
    return categorized


def format_findings(cat):
    segs = []
    if cat.get("positive"):
        segs.append("Positive findings: " + ", ".join(cat["positive"]))
    if cat.get("negative"):
        segs.append("Negative findings: " + ", ".join(cat["negative"]))
    if cat.get("uncertain"):
        segs.append("Uncertain findings: " + ", ".join(cat["uncertain"]))
    return ". ".join(segs) if segs else "no common findings"


THRESHOLD_PATH = cfg.config.model.mhcac.threshold_path
PROMPT_PREFIX = "Image information: " + ("<IMG>" * 32)
PROMPT_TAIL = (
    "\n\nAct as an expert radiologist. Using only the structured abnormality information and the image-derived "
    "features above, write the *Findings* section of a chest X-ray report.\n\n"
    "- Do not invent findings. Only describe abnormalities explicitly provided in the 'Abnormality information'.\n"
    "- Do not repeat the same information using different wording.\n"
    "- Use a single, fluent paragraph in formal radiological style.\n"
    "- Use cautious and precise language if uncertain abnormalities are present.\n"
    "- Avoid enumeration, bullet points, and speculative phrases.\n"
    "- The report should reflect the clinical tone and structure of professionally written reports.\n\n"
    "Return only the generated findings text."
)
SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant."
    "The assistant gives professional, detailed, and polite answers to the user's questions."
)


def build_prompt(findings_text):
    user_msg = f"{PROMPT_PREFIX}.\n\nAbnormality information: {findings_text}{PROMPT_TAIL}"
    return f"{SYSTEM} USER: {user_msg} ASSISTANT:"


dataset = MIMIC_CXR_Dataset(
    vis_processor=None, text_processor=None,
    vis_root=MIMIC_IMG_ROOT,
    split=EVAL_SPLIT, cfg=cfg, truncate=None,
)
print(f"{EVAL_SPLIT} split size: {len(dataset)}")

loader = DataLoader(
    dataset, batch_size=BATCH_SIZE_IMG, shuffle=False,
    num_workers=2, pin_memory=True,
)

PREDS_PATH = os.path.join(OUT_DIR, f"predictions_{EVAL_SPLIT}.json")
predictions, all_logits, all_labels = [], [], []
processed = 0
t0 = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="Inference"):
        if EVAL_LIMIT is not None and processed >= EVAL_LIMIT:
            break
        images = batch["image"].cuda(non_blocking=True)
        out = blip_model.forward_image(images)
        logits, qformer_embs = out[0], out[1]
        all_logits.append(logits.detach().cpu())
        all_labels.append(batch["classification_labels"])

        for j in range(images.size(0)):
            if EVAL_LIMIT is not None and processed >= EVAL_LIMIT:
                break
            dicom_id = batch["dicom_id"][j]
            cat = classify_abnormalities(logits[j].detach().cpu(), thresholds_path=THRESHOLD_PATH)
            findings_text = format_findings(cat)
            prompt = build_prompt(findings_text)

            # Mirror inference.py: dump current image embeds; model reads them via use_img=True
            torch.save(qformer_embs[j].unsqueeze(0).detach().cpu(), "current_chat_img.pt")

            inputs = vicuna_tokenizer(prompt, return_tensors="pt")
            input_ids = inputs["input_ids"].to(lang_model.device)
            gen = lang_model.generate(
                input_ids=input_ids,
                dicom=[dicom_id],
                use_img=True,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )
            decoded = vicuna_tokenizer.batch_decode(gen.sequences, skip_special_tokens=True)[0]
            pred_text = decoded.split("ASSISTANT:")[-1].strip()

            predictions.append({
                "image_id": int(dataset.img_ids[dicom_id]),
                "dicom_id": str(dicom_id),
                "caption": pred_text,
                "gt_findings": batch["text_output"][j] if "text_output" in batch else "",
                "findings_summary": findings_text,
            })
            processed += 1

            if processed % 100 == 0:
                with open(PREDS_PATH, "w") as f:
                    json.dump(predictions, f, ensure_ascii=False, indent=2)

with open(PREDS_PATH, "w") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
elapsed = time.time() - t0
print(f"Generated {len(predictions)} reports in {elapsed:.1f}s ({elapsed/max(len(predictions),1):.2f}s/sample)")
print(f"Predictions saved -> {PREDS_PATH}")

# Save logits for CheXpert F1
ALL_LOGITS = torch.cat(all_logits, dim=0) if all_logits else torch.empty(0)
ALL_LABELS = torch.cat(all_labels, dim=0) if all_labels else torch.empty(0)
if EVAL_LIMIT is not None:
    ALL_LOGITS = ALL_LOGITS[:EVAL_LIMIT]
    ALL_LABELS = ALL_LABELS[:EVAL_LIMIT]
torch.save({"logits": ALL_LOGITS, "labels": ALL_LABELS}, os.path.join(OUT_DIR, "classification.pt"))
print(f"Saved classification tensor: logits {tuple(ALL_LOGITS.shape)}  labels {tuple(ALL_LABELS.shape)}")


In [ ]:
# Cell 9 — Compute BLEU-1/2/3/4 + METEOR + ROUGE-L via MIMICEvalCap
import os, json
from model.lavis.data.ReportDataset import MIMICEvalCap

gts_df = dataset.annotation[["dicom_id", "findings"]].copy()
evaluator = MIMICEvalCap(gts=gts_df, img_id_map=dataset.img_ids)
scores, gts_img_id = evaluator.evaluate(predictions)

print("\n=== Report-generation metrics ===")
for k in ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "agg_metrics"]:
    if k in scores:
        print(f"  {k:14s}: {scores[k]:.4f}")

scores_path = os.path.join(OUT_DIR, "scores.json")
with open(scores_path, "w") as f:
    json.dump({k: float(v) for k, v in scores.items()}, f, indent=2)
print(f"Saved -> {scores_path}")


In [ ]:
# Cell 10 — CheXpert-style classification F1 over 14 abnormalities
import os, json
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

if RUN_CHEXPERT_F1 and ALL_LOGITS.numel() > 0:
    # Approach A: argmax over 3-way logits per abnormality, binary positive vs rest
    preds_cls  = ALL_LOGITS.argmax(dim=-1).numpy()    # (N, 14)
    labels_cls = ALL_LABELS.numpy()                    # (N, 14)
    assert preds_cls.shape == labels_cls.shape, f"Shape mismatch: {preds_cls.shape} vs {labels_cls.shape}"

    per_task = {}
    for t, name in enumerate(ABN_NAMES):
        y_true = (labels_cls[:, t] == 1).astype(int)
        y_pred = (preds_cls[:, t] == 1).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        per_task[name] = {
            "precision": float(p), "recall": float(r), "f1": float(f1), "acc": float(acc),
            "n_pos_true": int(y_true.sum()), "n_pos_pred": int(y_pred.sum()),
        }

    macro_p  = float(np.mean([v["precision"] for v in per_task.values()]))
    macro_r  = float(np.mean([v["recall"]    for v in per_task.values()]))
    macro_f1 = float(np.mean([v["f1"]        for v in per_task.values()]))

    print("\n=== CheXpert classification F1 (positive class) ===")
    header = f"{'Abnormality':30s}  Prec   Recall  F1     n_pos_true  n_pos_pred"
    print(header)
    print("-" * len(header))
    for name in ABN_NAMES:
        v = per_task[name]
        print(f"{name:30s}  {v['precision']:.3f}  {v['recall']:.3f}  {v['f1']:.3f}"
              f"  {v['n_pos_true']:5d}       {v['n_pos_pred']:5d}")
    print(f"\nMacro avg -> Prec: {macro_p:.4f}  Recall: {macro_r:.4f}  F1: {macro_f1:.4f}")

    chex_path = os.path.join(OUT_DIR, "chexpert_f1.json")
    with open(chex_path, "w") as f:
        json.dump({
            "per_task": per_task,
            "macro": {"precision": macro_p, "recall": macro_r, "f1": macro_f1},
        }, f, indent=2)
    print(f"Saved -> {chex_path}")
else:
    print("Skipped CheXpert F1 (RUN_CHEXPERT_F1=False or no logits collected)")

# Approach B - Java CheXpert labeler on generated reports (optional, fragile)
if TRY_TEXT_LABELER:
    print("\nWARN: Java-based CheXpert text labeler is NOT bundled in this notebook.")
    print("  To enable: attach a chexpert-labeler Kaggle dataset (or clone https://github.com/stanfordmlgroup/chexpert-labeler)")
    print("  then implement subprocess invocation here. Approach A above is the primary metric.")


In [ ]:
# Cell 11 — (Optional) Push evaluation outputs back to a Kaggle dataset version
import os, json, shutil, subprocess, time

if HAVE_KAGGLE_CREDS:
    PUSH_DIR = "/kaggle/working/eval_results_push"
    os.makedirs(PUSH_DIR, exist_ok=True)

    result_files = [
        f"predictions_{EVAL_SPLIT}.json",
        "scores.json",
        "chexpert_f1.json",
        "classification.pt",
    ]
    for fname in result_files:
        src = os.path.join(OUT_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, PUSH_DIR)

    DATASET_SLUG = f"{KAGGLE_USER}/meta-cxr-eval-results"
    meta = {
        "title": "META-CXR evaluation outputs",
        "id": DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(PUSH_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(meta, f)

    timestamp = time.strftime("%Y-%m-%d %H:%M")
    try:
        subprocess.check_call(
            ["kaggle", "datasets", "version", "-p", PUSH_DIR, "-m", f"Eval run {timestamp}"]
        )
        print(f"OK - Pushed new version of {DATASET_SLUG}")
    except subprocess.CalledProcessError:
        try:
            subprocess.check_call(["kaggle", "datasets", "create", "-p", PUSH_DIR])
            print(f"OK - Created new dataset {DATASET_SLUG}")
        except subprocess.CalledProcessError as e:
            print(f"WARN - Push failed: {e}. Files remain at {PUSH_DIR}.")
else:
    print("Skipped Kaggle push (no credentials configured in Cell 2).")
